# Motion energy — example analysis

Demonstrates the `aind-motion-energy` pipeline on a behavior video and aligns the
resulting trace to behavioral events (go cues) from the session NWB.

**Time reference used throughout**

| Clock | Definition |
|---|---|
| `behavior_time` | Harp/hardware acquisition seconds — what NWB stores |
| frame | Video frame index: row i of the companion CSV and of the ME trace |
| `session_time` | Seconds from the first go cue (first go cue = 0.0 s) |

Run this in Code Ocean JupyterLab with the behavior data asset and
`foraging_nwb_bonsai` data asset both attached.

In [ ]:
%matplotlib inline
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pynwb
from aind_dynamic_foraging_behavior_video_analysis import video_timing_qc as vtq
from aind_dynamic_foraging_behavior_video_analysis.video_alignment import behavior_time_to_frame_index
from aind_motion_energy import clean_trace, compute_motion_energy, render_motion_energy_video
from aind_motion_energy.io import get_video_info

DATA_DIR    = Path("/root/capsule/data")
NWB_DIR     = DATA_DIR / "foraging_nwb_bonsai"
SCRATCH_DIR = Path("/root/capsule/scratch")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mkv", ".mov", ".mj2"}

## 1. Discover data assets

In [ ]:
# List behavior video files (excludes foraging_nwb_bonsai tree)
behavior_assets = [p for p in DATA_DIR.iterdir() if p.is_dir() and p.name != "foraging_nwb_bonsai"]
print("Behavior assets:", [a.name for a in behavior_assets])

videos = sorted(
    p for a in behavior_assets
    for p in a.rglob("*") if p.suffix.lower() in VIDEO_EXTS
)
for v in videos:
    info = get_video_info(v)
    print(f"  {v.relative_to(DATA_DIR)}  {info['n_frames']} frames @ {info['fps']} fps  ({info['width']}×{info['height']})")

## 2. Select video and find companion CSV + NWB

The companion CSV sits alongside the video and records a `Behav_Time` (Harp) timestamp
for every frame — this is what anchors the video to the behavior clock. The raw column
is late after dropped frames, so Section 5 corrects it before use.

The NWB is located in `foraging_nwb_bonsai/` in a session folder that shares the
subject-ID and date with the behavior asset mount name (`behavior_{subject}_{date}_*`).

In [ ]:
VIDEO = videos[0]    # change index to pick a different camera
print("Using video:", VIDEO.relative_to(DATA_DIR))

# Companion CSV: same folder as the video. Either AIND layout is
# handled by the library: headered <Camera>/metadata.csv, or
# headerless <camera>.csv.
csv_candidates = sorted(VIDEO.parent.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV found alongside {VIDEO}")
VIDEO_CSV = csv_candidates[0]
print("Companion CSV:", VIDEO_CSV.relative_to(DATA_DIR))

# Match NWB by subject ID + date extracted from the asset mount name
# e.g. "behavior_854151_2026-06-08_09-39-38" → subject="854151", date="2026-06-08"
asset_name = VIDEO.relative_to(DATA_DIR).parts[0]
m = re.match(r"behavior_(\d+)_(\d{4}-\d{2}-\d{2})", asset_name)
if not m:
    raise ValueError(f"Cannot parse subject/date from asset name: {asset_name}")
subject, date = m.group(1), m.group(2)
print(f"Subject: {subject}  Date: {date}")

nwb_candidates = sorted(NWB_DIR.rglob("*.nwb"))
nwb_candidates = [f for f in nwb_candidates if subject in f.parts[-2] and date in f.parts[-2]]
if not nwb_candidates:
    # fallback: any NWB under a folder containing subject + date anywhere
    nwb_candidates = sorted(f for f in NWB_DIR.rglob("*.nwb") if subject in str(f) and date in str(f))
if not nwb_candidates:
    raise FileNotFoundError(f"No NWB found for subject {subject} date {date} under {NWB_DIR}")
NWB_PATH = nwb_candidates[0]
print("NWB:", NWB_PATH.relative_to(DATA_DIR))

## 3. Compute motion energy

In [ ]:
END_FRAME = 400000   # set to None for the full recording

result = compute_motion_energy(VIDEO, end_frame=END_FRAME)
# Traces are frame-aligned: row i is video frame i, and row 0 is NaN (no
# preceding frame). Column 0 is the whole frame; pass rois= (see
# load_roi_masks) to get one more column per ROI, named by result.roi_ids.
me = result.me[:, 0]
keyframe_mask = result.keyframe_mask
avg_map = result.avg_map
meta = result.metadata
me_clean = clean_trace(me, keyframe_mask)
fps = meta["fps"]

print(f"{meta['n_keyframes_masked']} keyframe diffs masked of {len(me)} total")
print(f"Mean ME = {np.nanmean(me_clean):.4f}")

## 4. Static summary

In [ ]:
t = np.arange(len(me)) / fps

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

axes[0].plot(t, me,       lw=0.4, color="0.78", alpha=0.8)
axes[0].plot(t, me_clean, lw=0.7, color="steelblue")
axes[0].set_xlabel("Time from video start (s)")
axes[0].set_ylabel("Motion energy")
axes[0].set_title(VIDEO.stem)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

im = axes[1].imshow(avg_map, cmap="hot", aspect="auto")
plt.colorbar(im, ax=axes[1], label="Mean abs diff (per pixel)")
axes[1].set_title("Average motion map (keyframe-excluded)")

fig.tight_layout()
plt.show()

## 5. Load NWB and compute time alignment

`goCue_start_time` in the NWB trials table is in **behavior_time** (Harp clock).
`behavior_time_to_frame_index()` finds the first video frame at or after each go cue on
the **corrected** per-frame Harp time from `video_timing_qc` (with the session's trigger
log when present). Frame i is row i of the ME trace, so go cues index ME directly. The raw
CSV `Behav_Time` (and seconds-into-the-video from it) is late after dropped frames.

In [ ]:
io  = pynwb.NWBHDF5IO(str(NWB_PATH), "r")
nwb = io.read()

trials = nwb.trials.to_dataframe()
print(f"{len(trials)} trials total")

# goCue_start_time is the AIND standard name; it is in behavior_time (Harp)
gocue_behavior_times = trials["goCue_start_time"].dropna().values
print(f"First 3 go-cue behavior_times: {gocue_behavior_times[:3]}")

In [ ]:
# Corrected behavior_time of every video frame (row i = frame i = ME row i).
# Raises for sessions whose timing cannot be trusted (e.g. a Harp clock step).
trigger_log = DATA_DIR / asset_name / "behavior" / "raw.harp" / "BehaviorEvents" / "Event_94.bin"
trigger_times = vtq.read_harp_trigger_log(trigger_log) if trigger_log.exists() else None
timing = vtq.correct_video_timing(vtq.load_video_timing(VIDEO_CSV), trigger_times=trigger_times)
harp_time = timing["harp_time"].to_numpy()
print(f"First frame behavior_time: {harp_time[0]:.3f} s  (trigger log: {trigger_times is not None})")

# go cue behavior_time → first frame at or after it, i.e. its ME row.
# Drop go cues before the first frame or past the processed clip.
in_video = gocue_behavior_times >= harp_time[0]
gocue_frames = behavior_time_to_frame_index(gocue_behavior_times[in_video], harp_time)
gocue_frames = gocue_frames[gocue_frames < len(me)]
print(f"{len(gocue_frames)} go cues fall within the processed clip")

## 6. Trial-aligned motion energy

Each row of the heatmap is a single trial; time 0 = go-cue onset.

In [ ]:
PRE_S  = 0.5   # seconds before go cue
POST_S = 2.0   # seconds after go cue

pre_fr  = int(PRE_S  * fps)
post_fr = int(POST_S * fps)

aligned = []
for f0 in gocue_frames:
    s, e = f0 - pre_fr, f0 + post_fr
    if 0 <= s and e <= len(me_clean):
        aligned.append(me_clean[s:e])

if not aligned:
    print("No trials with a full window inside the clip — try a larger END_FRAME.")
else:
    aligned = np.array(aligned)
    trial_t = np.linspace(-PRE_S, POST_S, aligned.shape[1])
    print(f"{len(aligned)} complete trials")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    im = axes[0].imshow(
        aligned, aspect="auto",
        extent=[-PRE_S, POST_S, len(aligned), 0],
        cmap="viridis",
    )
    axes[0].axvline(0, color="white", lw=0.8, ls="--")
    axes[0].set_xlabel("Time from go cue (s)")
    axes[0].set_ylabel("Trial")
    axes[0].set_title("ME per trial")
    plt.colorbar(im, ax=axes[0])

    mean = np.nanmean(aligned, axis=0)   # NaN-aware: frame 0 is NaN
    sem  = np.nanstd(aligned, axis=0) / np.sqrt(len(aligned))
    axes[1].fill_between(trial_t, mean - sem, mean + sem, alpha=0.25, color="steelblue")
    axes[1].plot(trial_t, mean, color="steelblue", lw=1.2)
    axes[1].axvline(0, color="0.4", lw=0.8, ls="--", label="go cue")
    axes[1].set_xlabel("Time from go cue (s)")
    axes[1].set_ylabel("Motion energy")
    axes[1].set_title(f"Mean ± SEM  (n={len(aligned)} trials)")
    axes[1].legend(fontsize=8)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    fig.tight_layout()
    plt.show()

## 7. Rendered example clip

Synced video + scrolling ME plot for a 4-second window around the first go cue.
Saved to `/root/capsule/scratch/`.

In [ ]:
if len(gocue_frames):
    f0 = gocue_frames[5]
    clip_start = max(0, f0 - int(1.0 * fps))    # 1 s before go cue
    clip_end   = f0 + int(3.0 * fps)            # 3 s after go cue

    # me/me_clean are frame-aligned (row i = frame i, see Section 3), so the
    # same [clip_start:clip_end] slice gives one row per rendered frame.
    out_path = SCRATCH_DIR / f"{VIDEO.stem}_go_cue_example.mp4"
    render_motion_energy_video(
        VIDEO, me_clean[clip_start:clip_end],
        fps_source=fps,
        output_path=out_path,
        raw_trace=me[clip_start:clip_end],
        start_frame=clip_start,
        end_frame=clip_end,
        window_seconds=2.0,
        out_fps=120.0,
    )
    print(f"Saved: {out_path}")